In [ ]:
!pip install langchain langchain-community transformers huggingface_hub pypdf python-docx

In [ ]:
from google.colab import files
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from pypdf import PdfReader
import docx


In [ ]:
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
print('Uploaded:', file_name)

In [ ]:
def extract_text(file_name):
    text = ''
    if file_name.endswith('.pdf'):
        reader = PdfReader(file_name)
        for page in reader.pages:
            text += page.extract_text() or ''
    elif file_name.endswith('.docx'):
        doc = docx.Document(file_name)
        for para in doc.paragraphs:
            text += para.text + '\n'
    return text

resume_text = extract_text(file_name)
print(resume_text[:1000])

In [ ]:
job_desc = input('Paste Job Description: ')

In [ ]:
pipe = pipeline('text-generation', model='tiiuae/falcon-7b-instruct', max_new_tokens=300)
llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
template = '''
You are a professional resume analyzer.
Analyze the resume based on the job description.

Resume:
{resume}

Job Description:
{jd}

Provide:
- Skills
- Strengths
- Weaknesses
- Match Score (/100)
- Suggestions
'''

prompt = PromptTemplate(input_variables=['resume','jd'], template=template)
chain = LLMChain(llm=llm, prompt=prompt)

In [ ]:
result = chain.invoke({'resume': resume_text[:1500], 'jd': job_desc})
print('===== RESULT =====')
print(result['text'])